# Neural Prototyping

# Google Colab Mounting

In [113]:
!rm -rf /content/credit-risk-modeling
!git clone https://github.com/BillyBrothers/credit-risk-modeling.git
!pip install -r /content/credit-risk-modeling/requirements.txt

import sys 
sys.path.append('/content/credit-risk-modeling')

Cloning into 'credit-risk-modeling'...
remote: Enumerating objects: 1271, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 1271 (delta 106), reused 91 (delta 47), pack-reused 1121 (from 1)
Receiving objects: 100% (1271/1271), 52.36 MiB | 17.44 MiB/s, done.
Resolving deltas: 100% (868/868), done.
Updating files: 100% (68/68), done.


In [114]:
!pip install scikeras

In [115]:
pip install -q -U keras-tuner

In [116]:
# data analysis
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pyampute.exploration.md_patterns import mdPatterns
from pyampute.exploration.mcar_statistical_tests import MCARTest
import missingno as msno

# preprocessing
import sklearn.utils.validation
import sys
from scipy import stats
from scipy.stats import shapiro, distributions, loguniform
from scipy.stats.mstats import winsorize
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, HalvingRandomSearchCV, HalvingGridSearchCV, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, QuantileTransformer, MinMaxScaler, KBinsDiscretizer, Binarizer, PolynomialFeatures, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from feature_engine.outliers import Winsorizer
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.pipeline import Pipeline
from sklearn import set_config

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# Modeling
from sklearn.linear_model import RidgeClassifier, LogisticRegression, RidgeClassifierCV, LogisticRegressionCV, SGDClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
import joblib

# Metrics
from sklearn.metrics import confusion_matrix, recall_score, precision_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report, precision_recall_curve, PrecisionRecallDisplay, log_loss, brier_score_loss, roc_curve, roc_auc_score, RocCurveDisplay, det_curve, DetCurveDisplay, fbeta_score, average_precision_score, matthews_corrcoef

# Calibration
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV

# Inspection
from sklearn.inspection import PartialDependenceDisplay
from credit_risk_modeling import model_eval

import tensorflow as tf
from tensorflow import keras
from keras import layers
from scikeras.wrappers import KerasClassifier
import keras_tuner as kt

In [117]:
!ls

credit-risk-modeling  logs  sample_data


In [118]:
!ls credit-risk-modeling

credit_risk_modeling  LICENSE	 pyproject.toml  requirements.txt
data		      Makefile	 README.md	 tests
docs		      models	 references
environment.yml       notebooks  reports


# Imports

In [119]:
X_train = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_train_neural.csv"
)

In [120]:
X_test = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_test_neural.csv"
)

In [121]:
X_val = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_val_neural.csv"
)

In [122]:
y_train= pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_train.csv"
)
y_train = y_train.values.ravel()
neg, pos = np.bincount(y_train)
total = neg + pos
print(f"Examples:\n The training sets total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The training sets total amount of samples: 22686
 Positive: 4962 (21.87% of total)


In [123]:
y_test = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_test.csv"
)
y_test = y_test.values.ravel()
neg, pos = np.bincount(y_test)
total = neg + pos
print(f"Examples:\n The testing set total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The testing set total amount of samples: 2917
 Positive: 638 (21.87% of total)


In [124]:
y_val = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_val.csv"
)
y_val = y_val.values.ravel()
neg, pos = np.bincount(y_val)
total = neg + pos
print(f"Examples:\n The validation set total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The validation set total amount of samples: 6806
 Positive: 1488 (21.86% of total)


# Build Sequential Models

In [125]:
# # Architecture 1: Single hidden dense layer
def mlp1():
    model = keras.Sequential(name='MLP-1')
    model.add(keras.Input(shape=(X_train.shape[1], ))),
    model.add(layers.Dense(units=64, activation='relu')),
    model.add(layers.Dropout(rate= 0.20)),
    model.add(layers.Dense(units=1,activation='sigmoid')),
    model.compile(
        optimizer= keras.optimizers.Adam(),
        loss= keras.losses.BinaryCrossentropy(),
        metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
)
    return model

In [126]:
# def mlp1(hp):
#     model1 = keras.Sequential(name='MLP-1')
#     model1.add(keras.Input(shape=(X_train.shape[1], )))

#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     hp_lr = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')

#     model1.add(layers.Dense(units=hp_units, activation='relu'))
#     model1.add(layers.Dropout(rate= 0.20))
#     model1.add(layers.Dense(units=1,activation='sigmoid'))
    
#     model1.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_lr),
#                 loss=keras.losses.BinaryCrossentropy(),
#                 metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
#     ) 
#     return model1

In [127]:
# Architecture 2: Two hidden dense layers
def mlp2():
    model = keras.Sequential(name='MLP-2')
    model.add(keras.Input(shape=(X_train.shape[1], ))),
    model.add(layers.Dense(units=64, activation='relu')),
    model.add(layers.Dropout(rate= 0.20)),
    model.add(layers.Dense(units=128, activation='relu')),
    model.add(layers.Dropout(rate=0.20)),
    model.add(layers.Dense(units=1,activation='sigmoid')),
    
    model.compile(optimizer=keras.optimizers.Adam(),
                loss=keras.losses.BinaryCrossentropy(),
                metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
    )
    return model

In [128]:
# # Architecture 2: Two hidden dense layers
# def mlp2(hp):
#     model2 = keras.Sequential(name='MLP-2')
#     model2.add(keras.Input(shape=(X_train.shape[1], ))),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model2.add(layers.Dense(units=hp_units, activation='relu')),
#     model2.add(layers.Dropout(rate= 0.20)),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model2.add(layers.Dense(units=hp_units, activation='relu')),
#     model2.add(layers.Dropout(rate=0.20)),
#     model2.add(layers.Dense(units=1,activation='sigmoid')),
#     model2.compile(optimizer=keras.optimizers.Adam(),
#                 loss=keras.losses.BinaryCrossentropy(),
#                 metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
#     )
#     return model2

In [129]:
# Architecture 3: Three hidden dense layers
def mlp3():
    model = keras.Sequential(name='MLP-3')
    model.add(keras.Input(shape=(X_train.shape[1], ))),
    model.add(layers.Dense(units=64, activation='relu')),
    model.add(layers.Dropout(rate= 0.20)),
    model.add(layers.Dense(units=128, activation='relu')),
    model.add(layers.Dropout(rate=0.20)),
    model.add(layers.Dense(units=256, activation='relu')),
    model.add(layers.Dropout(rate=0.2)),
    model.add(layers.Dense(units=1,activation='sigmoid')),

    model.compile(
        optimizer= keras.optimizers.Adam(),
        loss= keras.losses.BinaryCrossentropy(),
        metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
)
    return model

In [130]:
# def mlp3(hp):
#     model3 = keras.Sequential(name='MLP-3')
#     model3.add(keras.Input(shape=(X_train.shape[1], ))),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model3.add(layers.Dense(units=hp_units, activation='relu')),
#     model3.add(layers.Dropout(rate= 0.20)),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model3.add(layers.Dense(units=hp_units, activation='relu')),
#     model3.add(layers.Dropout(rate=0.20)),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model3.add(layers.Dense(units=hp_units, activation='relu')),
#     model3.add(layers.Dropout(rate=0.2)),
#     model3.add(layers.Dense(units=1,activation='sigmoid')),
#     model3.compile(optimizer=keras.optimizers.Adam(),
#                 loss=keras.losses.BinaryCrossentropy(),
#                 metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
#     )

#     return model3

In [131]:
mlp_models = [
    ("MLP-1", mlp1()),
    ("MLP-2", mlp2()),
    ("MLP-3", mlp3())
]

### Callbacks

In [132]:
reduce_lr_plateau = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    verbose=1,
    min_lr=0.001
)

In [133]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    min_delta=1e-4,
    patience=7,
    verbose=1,
    restore_best_weights=True
)

In [134]:
log_dir = "logs/fit/"
tensorboard = keras.callbacks.TensorBoard(
    log_dir= log_dir
)

### Fit

In [135]:
class_weight = {
    0: 1.0,
    1: 2.0
    }

In [136]:
histories = {}

In [137]:
for model_name, model in mlp_models:
    print(f"Currently fitting model {model_name}.")
    history = model.fit(
        x= X_train,
        y= y_train,
        batch_size=32,
        epochs= 100,
        verbose=2,
        callbacks= [early_stopping, reduce_lr_plateau, tensorboard],
        validation_data= (X_val, y_val),
        class_weight= class_weight
    )
    histories[model_name] = history.history

Currently fitting model MLP-1.
Epoch 1/100
709/709 - 4s - 6ms/step - auc: 0.8418 - loss: 0.5669 - val_auc: 0.8894 - val_loss: 0.3434 - learning_rate: 1.0000e-03
Epoch 2/100
709/709 - 3s - 4ms/step - auc: 0.8893 - loss: 0.4776 - val_auc: 0.9015 - val_loss: 0.3265 - learning_rate: 1.0000e-03
Epoch 3/100
709/709 - 3s - 4ms/step - auc: 0.8962 - loss: 0.4612 - val_auc: 0.9050 - val_loss: 0.3167 - learning_rate: 1.0000e-03
Epoch 4/100
709/709 - 3s - 4ms/step - auc: 0.9002 - loss: 0.4497 - val_auc: 0.9067 - val_loss: 0.3164 - learning_rate: 1.0000e-03
Epoch 5/100
709/709 - 2s - 3ms/step - auc: 0.9018 - loss: 0.4438 - val_auc: 0.9091 - val_loss: 0.3054 - learning_rate: 1.0000e-03
Epoch 6/100
709/709 - 2s - 3ms/step - auc: 0.9051 - loss: 0.4347 - val_auc: 0.9099 - val_loss: 0.2955 - learning_rate: 1.0000e-03
Epoch 7/100
709/709 - 2s - 3ms/step - auc: 0.9064 - loss: 0.4296 - val_auc: 0.9114 - val_loss: 0.2915 - learning_rate: 1.0000e-03
Epoch 8/100
709/709 - 4s - 5ms/step - auc: 0.9071 - loss: 0

In [138]:
chosen_metric = 'val_auc'
max_auc_per_model = {}
best_auc = None
best_model = None
best_model_name = None

for model_name, model in mlp_models:
    max_auc_per_model[model_name] = max(histories[model_name][chosen_metric])
    max_auc_dict = dict([sorted(max_auc_per_model.items(), key= lambda item: item[1])[-1]])
    max_auc_model_name = list(max_auc_dict.keys())[0]
    max_aux_score = list(max_auc_dict.values())[0]
    if max_auc_model_name == model_name:
        best_auc = max_aux_score
        best_model_name = max_auc_model_name
        best_model = model
    else:
        continue

print(f"Results: \nBest model: {best_model_name}\nAUC score: {best_auc:.4f}")

Results: 
Best model: MLP-2
AUC score: 0.9257


In [139]:
best_model.summary()

Model: "MLP-2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_41 (Dense)                │ (None, 64)             │         1,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_27 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_42 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_28 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,189 (114.02 KB)

 Trainable params: 9,729 (38.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 19,460 (76.02 KB)

In [199]:
!ls credit-risk-modeling/models

best_distance_model.pkl     distance_preprocessed_pipeline.pkl
best_linear_model.pkl	    linear_preprocessed_pipeline.pkl
best_model.keras	    neural_preprocessed_pipeline.pkl
best_probability_model.pkl  probability_preprocessed_pipeline.pkl
best_tree_model.pkl	    tree_preprocessed_pipeline.pkl


In [195]:
pwd!

'/content'

In [196]:
best_model.save('credit-risk-modeling/models/best_model.keras')

### Hyperparameter Tuning

In [140]:
# for tid, t in tuner.oracle.trials.items():
#     print(tid, t.status, t.score)

In [141]:
# def mlp2_tuned(hp):
#     model = keras.Sequential(name='MLP-2-tuned')

#     units = hp.Int("units", 64, 256, step=32)
#     lr = hp.Float("learning_rate", 1e-4, 1e-2, sampling="log")

#     model.add(keras.Input(shape=(X_train.shape[1], ))),
#     model.add(layers.Dense(units=units, activation='relu')),
#     model.add(layers.Dropout(rate= 0.20)),
#     model.add(layers.Dense(units=units, activation='relu')),
#     model.add(layers.Dropout(rate=0.20)),
#     model.add(layers.Dense(units=1,activation='sigmoid')),
    
#     model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
#                 loss=keras.losses.BinaryCrossentropy(),
#                 metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
#     )
#     return model

In [142]:
# import shutil
# shutil.rmtree("untitled_project", ignore_errors=True)

In [143]:
# tuner = kt.Hyperband(
#     hypermodel= mlp2_tuned,
#     objective="val_auc",
#     max_epochs= 50,
#     factor=3,
# )

In [144]:
# tuner.search_space_summary()

In [145]:
# tuner.search(
#     X_train,
#     y_train,
#     validation_data= (X_val, y_val),
#     callbacks= [early_stopping]
# )

In [146]:
# tuner.results_summary()

### Calibration

Scikeras cannot accept an instance of a model only a user building function (factory function). So, my model will have NO weights on it.

In [147]:
models_only = {}
mlp_functions = []

In [148]:
for model_name, model in mlp_models:
    models_only[model_name] = KerasClassifier(
    model= model,
    optimizer= keras.optimizers.Adam(),
    loss= keras.losses.BinaryCrossentropy(),
    random_state=42,
    class_weight= class_weight,
    metrics= ['val_auc'],
    callbacks= [early_stopping, reduce_lr_plateau],
    validation_split= 0.20,
    epochs=100
)

In [149]:
for model_name, model in list(models_only.items()):
    mlp_functions.append(model)

In [150]:
models_performances, fitted_models = model_eval.comparing_models(
    mlp_functions,
    X_train,
    y_train,
    X_test,
    y_test
)

Epoch 1/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - auc: 0.9261 - loss: 0.3574 - val_auc: 0.9291 - val_loss: 0.3617 - learning_rate: 0.0010
Epoch 2/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.9237 - loss: 0.3607 - val_auc: 0.9286 - val_loss: 0.3630 - learning_rate: 0.0010
Epoch 3/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - auc: 0.9240 - loss: 0.3621 - val_auc: 0.9282 - val_loss: 0.3634 - learning_rate: 0.0010
Epoch 4/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - auc: 0.9223 - loss: 0.3637 - val_auc: 0.9275 - val_loss: 0.3639 - learning_rate: 0.0010
Epoch 5/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.9247 - loss: 0.3585 - val_auc: 0.9276 - val_loss: 0.3640 - learning_rate: 0.0010
Epoch 6/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.9238 - loss: 0.3617 - val_auc: 0.9269 - val_loss: 0.3650 - learning_rate: 0.0010
Epoch 7/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.9259 - loss: 0.3569 - val_auc: 0.9272 - val_loss: 0.3659 - learning_rate: 0.0010

In [ ]:
fitted_models

In [178]:
fitted_models = pd.DataFrame(
    data=fitted_models,
    index= range(0, 1)
)

In [ ]:
fitted_models['MLP-2 (class_weight={0: 1.0, 1: 2.0})'].iloc[0]

In [152]:
roc_auc = models_performances.loc[:,'roc_auc']

In [153]:
clean_functions = {"mlp1": mlp1(), 
 "mlp2": mlp2(), 
 "mlp3": mlp3()},

In [154]:
clean_functions_df = pd.DataFrame(
    data= clean_functions,
    index= range(0,1)
)

In [155]:
clean_functions_df = pd.melt(
    frame=clean_functions_df,
    value_vars= list(clean_functions_df.columns),
    var_name= 'MLP',
    value_name= 'model'
)

In [156]:
clean_functions_df

,MLP,model
0,mlp1,"<Sequential name=MLP-1, built=True>"
1,mlp2,"<Sequential name=MLP-2, built=True>"
2,mlp3,"<Sequential name=MLP-3, built=True>"


In [157]:
clean_functions_df = pd.concat(
    objs= [clean_functions_df, roc_auc],
    axis=1
)

In [158]:
clean_functions_df.sort_values(
    by= 'roc_auc',
    axis=0,
    ascending=False,
    inplace= True
)

In [159]:
clean_functions_df.iloc[0]['model'].layers

[<Dense name=dense_50, built=True>,
 <Dropout name=dropout_33, built=True>,
 <Dense name=dense_51, built=True>,
 <Dropout name=dropout_34, built=True>,
 <Dense name=dense_52, built=True>]